In [ ]:
# uvicorn image_api.sql_app.main:app --port 8000 --reload
# uvicorn sql_app.main:app --port 8000 --reload
# uvicorn main:app --port 8000 --reload

from typing import List, Annotated
from fastapi import Depends, FastAPI, HTTPException, UploadFile, File, Form
import shapely as shp
from sqlalchemy.orm import Session
from sqlalchemy import text, select, MetaData, Table
import sqlalchemy as sqlA

from database import SessionLocal, engine
from crud_map import *
import models_map

import json
import os
import pandas as pd
import geopandas as gpd
import uvicorn

"""# RETRIEVES ALL IMPORTED MODULES
import sys
from pprint import pprint
pprint(list(sys.modules.keys())) # This line prints all imported modules
"""

In [2]:
#############################################################################################################################

# API parameters
cpu_count = 4
API_LINK = '127.0.0.5:8000'
app = FastAPI()

models_map.Base.metadata.create_all(bind=engine)


json_centroids = "C:/IMAGE/Foliage/image_api/preprocess/dataset_centroids.json"
url_db = "C:/IMAGE/Foliage/image_api/data/subgrids.db"


#############################################################################################################################

In [3]:
@app.post("/geodata/")
def read_coordinates(input_str: Annotated[str, Form()]):
    """
    Takes a comma-separated string as input and returns information of the region bounded by it.
    Input format: lon1,lat1,lon2,lat2,lon3,lat3,......
    """


    ################### (1) Create polygon object from input ###################
    polygon = read_coordinates(input_str)
    print("Step 1 Complete")


    ################### (2) Extract the geodata from SQL database ###################
    # Establish parameters for connection to the SQL database with all geodata
    list_closest_region = find_nearest_cluster(polygon, json_centroids)
    df = get_df_from_sql(url_db, list_closest_region[1])
    print("Step 2 Complete")


    ################### (3) Filter useful info to keep using polygon of bounded region ###################
    '''Detect the bounded area'''
    # Obtain data for overlapping region
    gdf_inPoly = filter_geodata(polygon, df)
    print("Step 3 Complete")


    ################### (4) API calls to get more detailed knowledge of bounded region ###################
    '''Percentage covers of relevant classes'''
    stats = get_stats(gdf_inPoly)
    print("Step 4 Complete")


    ################### (5) Display final output to user in desired format ###################
    # Convert the dataframe to JSON representation
    output = display_output(gdf_inPoly, stats)
    print("Step 5 Complete")


    ##########################################################################################
    return output


#############################################################################################################################

In [ ]:
if __name__ == '__main__': 
    host, port = API_LINK.split(':')
    uvicorn.run(
        "map:app",
        host=host,
        port=int(port),
        #factory=False,
        #reload=False,
        #loop='asyncio',
        workers=cpu_count,
    )